# Multiple Linear Regression — Compact Notebook

This notebook implements a minimal, step-by-step Multiple Linear Regression example.

**For comprehensive explanations, theory, and detailed examples**, see:
**[`../teaching/02_multiple_linear_regression.md`](../teaching/02_multiple_linear_regression.md)**

## Quick Steps Overview:

- **Step 1** — Import libraries & load data  
- **Step 2** — Read data and choose features & target  
- **Step 3** — Exploratory Data Analysis (EDA)  
- **Step 4** — Data cleaning: missing values, encoding, feature engineering  
- **Step 5** — Encode categorical variables (One-Hot Encoding)
- **Step 6** — Split data (Train / Test) and visualize split  
- **Step 7** — Train the model (Multiple Linear Regression)  
- **Step 8** — Make predictions and compare X_test vs y_test  
- **Step 9** — Evaluate performance (R², MAE, MSE)  
- **Step 10** — Visualize results and business insights


---

## Step 1: Import Libraries

| Library | Why we need it |
|---------|---------------|
| `numpy` | Numerical operations |
| `pandas` | Loading and manipulating the dataset |
| `matplotlib` | Plotting results |
| `seaborn` | Statistical visualisations (heatmaps, pairplots) |
| `warnings` | Suppress deprecation warnings from sklearn |

In [ ]:
# Import basic libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

## Step 2: Load the Dataset

The Startups dataset has **50 companies** with 5 columns:

| Column | Type | Description |
|--------|------|-------------|
| R&D Spend | Numerical | Research and development budget |
| Administration | Numerical | Administrative costs |
| Marketing Spend | Numerical | Marketing budget |
| State | Categorical | New York, California, or Florida |
| Profit | Numerical | Target variable — annual profit |

We use `iloc[:, :-1]` (all columns except the last) as features and `iloc[:, -1]` (the last column) as the target. This convention makes the code reusable across different datasets.

In [ ]:
# Load the dataset
dataset = pd.read_csv('../data/startups_data.csv')

print("Dataset loaded successfully!")
print("Dataset shape:", dataset.shape)
print("\nFirst 5 rows:")
print(dataset.head())

print("\nColumn names:")
print(dataset.columns.tolist())

print("\nDataset info:")
print(dataset.info())

# Define features and target
X = dataset.iloc[:, :-1].values  # All columns except last (features)
y = dataset.iloc[:, -1].values   # Last column (target: Profit)

print("\nFeatures (X) shape:", X.shape)
print("Target (y) shape:", y.shape)

print("\nFeature columns:", dataset.columns[:-1].tolist())
print("Target column:", dataset.columns[-1])

## Step 3: Exploratory Data Analysis (EDA)

EDA is not optional — it shapes every decision you make afterward.

**What to look for:**

- **Missing values** — do we need to impute? (None here, but always check)
- **Data types** — is `State` being read as a string? (Yes — needs encoding)
- **Scale differences** — does R&D Spend (0 to 165K) dwarf Administration (51K to 183K)? (Linear regression is scale-invariant, so this matters less than for KNN/SVM, but still worth knowing)
- **Correlations** — which features are most strongly correlated with Profit? The heatmap should show R&D Spend has the highest correlation, which we can verify with the model coefficients.
- **Distribution** — is the target (Profit) roughly normally distributed? Extreme skew can hurt regression performance.

In [ ]:
# Basic statistics
print("Dataset Description:")
print(dataset.describe())

print("\nMissing values:")
print(dataset.isnull().sum())

print("\nData types:")
print(dataset.dtypes)

print("\nUnique values in State column:")
print(dataset['State'].value_counts())

In [ ]:
# Visualize correlations (numerical columns only)
plt.figure(figsize=(10, 8))
numerical_data = dataset.select_dtypes(include=[np.number])
correlation_matrix = numerical_data.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

# Pairplot for numerical features
plt.figure(figsize=(12, 8))
sns.pairplot(numerical_data)
plt.suptitle('Pairplot of Numerical Features', y=1.02)
plt.tight_layout()
plt.show()

## Step 4: Data Cleaning

This step confirms what EDA revealed:
- No missing values — we can proceed without imputation
- One categorical column (`State`) — needs encoding before it can enter the regression equation
- Three numerical features already in usable form

In a real project, this step would also handle outlier treatment, feature engineering (e.g., creating ratio features like marketing-to-profit), and type corrections.

In [ ]:
# Check for missing values
print("Missing values per column:")
print(dataset.isnull().sum())

# Check data types
print("\nData types:")
print(dataset.dtypes)

# Identify categorical and numerical columns
categorical_columns = dataset.select_dtypes(include=['object']).columns.tolist()
numerical_columns = dataset.select_dtypes(include=[np.number]).columns.tolist()

print("\nCategorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)

# Display first few rows to understand the data structure
print("\nFirst 5 rows:")
print(dataset.head())

# Summary statistics for numerical columns
print("\nSummary statistics for numerical features:")
print(dataset[numerical_columns].describe())

## Step 5: One-Hot Encode the State Column

Linear regression computes: `Profit = b0 + b1*X1 + b2*X2 + ...`

This only works with numbers. `State` contains strings. We have two encoding choices:

**Why NOT label encoding (0, 1, 2)?**
Encoding New York=0, California=1, Florida=2 implies that California is twice New York and Florida is their average. These arithmetic relationships do not exist — they are just three different states.

**Why one-hot encoding?**
Creates a separate binary column for each state — no false ordering:
```
State=New York    →   [1, 0, 0]
State=California  →   [0, 1, 0]
State=Florida     →   [0, 0, 1]
```

**Dummy variable trap:** With 3 states, we only need 2 columns. If both `California=0` and `Florida=0`, we know the company is from New York — the third column is perfectly predictable from the other two. Including all three creates **perfect multicollinearity**, which makes the matrix non-invertible. We drop the first column (`X = X[:, 1:]`) to avoid this.

In [ ]:
# Import encoding libraries
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Apply One-Hot Encoding to categorical variables
# We need to encode the 'State' column (which is in column index 3)

# Create column transformer for one-hot encoding
ct = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(), [3])  # Apply OneHotEncoder to column 3 (State)
    ], 
    remainder='passthrough'  # Keep other columns as they are
)

# Apply the transformation
X = ct.fit_transform(X)

print("Features after One-Hot Encoding:")
print("Shape:", X.shape)
print("\nFirst 5 rows:")
print(X[:5])

# Note: After one-hot encoding, we typically need to avoid the dummy variable trap
# by dropping one of the encoded columns. We'll handle this by removing the first column.
X = X[:, 1:]

print("\nAfter avoiding dummy variable trap:")
print("Shape:", X.shape)
print("\nFirst 5 rows:")
print(X[:5])

## Step 6: Train/Test Split

We split 80% for training (40 companies) and 20% for testing (10 companies).

With only 50 samples, a 10-sample test set produces a noisy performance estimate. In practice, cross-validation would be preferred over a single split for such a small dataset — each test sample has an outsized effect on the final R² score.

The distribution plots confirm both splits draw from the full range of profit values — neither is accidentally all high-profit or all low-profit companies.

In [ ]:
# Import train_test_split
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print("Data split successfully!")
print("Training set - Features:", X_train.shape, "Target:", y_train.shape)
print("Testing set - Features:", X_test.shape, "Target:", y_test.shape)



In [ ]:
# Visualize the split distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Training set distribution
axes[0].hist(y_train, bins=15, alpha=0.7, color='blue', edgecolor='black')
axes[0].set_title('Training Set - Profit Distribution')
axes[0].set_xlabel('Profit')
axes[0].set_ylabel('Frequency')

# Testing set distribution
axes[1].hist(y_test, bins=15, alpha=0.7, color='red', edgecolor='black')
axes[1].set_title('Testing Set - Profit Distribution')
axes[1].set_xlabel('Profit')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print("✅ Data split visualization completed!")
print(f"Training set: {len(y_train)} samples")
print(f"Testing set: {len(y_test)} samples")

## Step 7: Train Multiple Linear Regression

Multiple Linear Regression extends simple linear regression to multiple features:

$$\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + b_3 x_3 + b_4 x_4 + b_5 x_5$$

Where:
- $b_0$ = intercept (baseline profit with all features at zero)
- $b_1, b_2$ = coefficients for the two state dummy variables
- $b_3, b_4, b_5$ = coefficients for R&D Spend, Administration, Marketing Spend

**How does sklearn find these coefficients?**

It minimises the sum of squared residuals using the **closed-form Ordinary Least Squares (OLS) solution**:

$$\mathbf{b} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

Unlike gradient descent (used in neural networks), OLS finds the exact optimal solution in one matrix operation — no learning rate, no epochs needed.

**What the coefficients tell you:**
The R&D Spend coefficient (~0.77) means: holding all other features constant, every additional dollar in R&D spend increases profit by ~$0.77. Read the printed equation carefully — the R&D coefficient is the most economically meaningful.

In [ ]:
# Import LinearRegression
from sklearn.linear_model import LinearRegression

# Create and train the Multiple Linear Regression model
regressor = LinearRegression()
regressor.fit(X_train, y_train)

print("Multiple Linear Regression model trained successfully!")
print("\nModel coefficients:")
print("Intercept (b0):", regressor.intercept_)
print("Coefficients (b1, b2, ...):", regressor.coef_)

# Display the linear equation
print("\nLinear Equation:")
equation = f"y = {regressor.intercept_:.2f}"
for i, coef in enumerate(regressor.coef_):
    equation += f" + ({coef:.2f}) * x{i+1}"
print(equation)

## Step 8: Make Predictions

The comparison table shows actual vs predicted profit for each of the 10 test companies.

**What you are looking for:**
- Are predictions systematically too high or too low? (Bias — would show up as consistently positive or negative differences)
- Are errors roughly proportional to the actual profit level? (Heteroscedasticity — important for regression assumption checking)
- Are there any outliers with unusually large errors?

A quick scan of the table gives intuition that the formal metrics in the next step will quantify.

In [ ]:
# Make predictions on the test set
y_pred = regressor.predict(X_test)

print("Predictions made successfully!")
print("Number of predictions:", len(y_pred))

# Compare actual vs predicted values
comparison_df = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred,
    'Difference': y_test - y_pred,
    'Absolute_Difference': np.abs(y_test - y_pred)
})

print("\nActual vs Predicted comparison:")
print(comparison_df)

# Display some statistics about predictions
print("\nPrediction Statistics:")
print("Mean Absolute Error in comparison:", comparison_df['Absolute_Difference'].mean())
print("Max difference:", comparison_df['Absolute_Difference'].max())
print("Min difference:", comparison_df['Absolute_Difference'].min())

## Step 9: Evaluate Model Performance

**R² (Coefficient of Determination):**
$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

Measures what fraction of the variance in Profit is explained by the features. R²=0.93 means the model explains 93% of profit variation — the remaining 7% is driven by factors not in our dataset.

**MAE (Mean Absolute Error):** Average dollar error in the predictions. Interpretable — same units as Profit. "On average, the model is off by $7,500."

**RMSE (Root Mean Squared Error):** Like MAE but penalises large errors more. If RMSE >> MAE, you have a few predictions with very large errors.

**Comparing training vs test R²:**
- Training R²=0.95, Test R²=0.93 → small gap, no significant overfitting
- If the gap were 0.95 vs 0.60, that would indicate the model memorised training patterns that do not generalise

**Benchmark:** A model that always predicts the mean profit would get R²=0. Our R²=0.93 is strong for a simple linear model on 50 samples.

In [ ]:
# Import metrics
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Calculate performance metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("=== MODEL PERFORMANCE METRICS ===")
print(f"R² Score: {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

print(f"\nR² Interpretation: {r2*100:.2f}% of the variance in profit is explained by our features")

# Training set performance for comparison
y_train_pred = regressor.predict(X_train)
r2_train = r2_score(y_train, y_train_pred)
print(f"\nTraining R² Score: {r2_train:.4f}")
print(f"Testing R² Score: {r2:.4f}")

if r2_train - r2 > 0.1:
    print("Warning: Possible overfitting detected (large gap between training and testing performance)")
else:
    print("Good: No significant overfitting detected")

## Step 10: Visualise Results and Business Insights

**The four plots tell a complete story:**

**1. Actual vs Predicted (top left)**
Points close to the diagonal line indicate accurate predictions. Systematic deviation above or below the line reveals bias.

**2. Residuals Plot (top right)**
Residuals (actual minus predicted) should be randomly scattered around zero. Patterns here violate the linear regression assumption of homoscedasticity:
- Funnel shape (residuals growing with predicted value) → variance is not constant
- Curved pattern → the true relationship is non-linear

**3. Residual Distribution (bottom left)**
Residuals should be approximately normally distributed (bell-shaped) for inference (confidence intervals, p-values) to be valid.

**4. Feature Coefficients (bottom right)**
The bar chart shows the model's raw coefficients. The most important business insight: which features drive profit most?

**Key business insight:** The R&D Spend coefficient is by far the most influential feature. Every dollar invested in R&D generates approximately $0.77 in profit. This is the kind of decision-relevant finding that makes regression valuable beyond pure prediction.

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Actual vs Predicted scatter plot
axes[0, 0].scatter(y_test, y_pred, alpha=0.7, color='blue')
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Profit')
axes[0, 0].set_ylabel('Predicted Profit')
axes[0, 0].set_title('Actual vs Predicted Profit')
axes[0, 0].grid(True, alpha=0.3)

# 2. Residuals plot
residuals = y_test - y_pred
axes[0, 1].scatter(y_pred, residuals, alpha=0.7, color='green')
axes[0, 1].axhline(y=0, color='r', linestyle='--')
axes[0, 1].set_xlabel('Predicted Profit')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title('Residuals Plot')
axes[0, 1].grid(True, alpha=0.3)

# 3. Distribution of residuals
axes[1, 0].hist(residuals, bins=15, alpha=0.7, color='orange', edgecolor='black')
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Residuals')
axes[1, 0].grid(True, alpha=0.3)

# 4. Feature importance (coefficients)
feature_names = ['State_California', 'State_Florida', 'R&D Spend', 'Administration', 'Marketing Spend']
coefficients = regressor.coef_
axes[1, 1].bar(range(len(coefficients)), coefficients, color='purple', alpha=0.7)
axes[1, 1].set_xlabel('Features')
axes[1, 1].set_ylabel('Coefficient Value')
axes[1, 1].set_title('Feature Coefficients (Importance)')
axes[1, 1].set_xticks(range(len(feature_names)))
axes[1, 1].set_xticklabels(feature_names, rotation=45, ha='right')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Business insights
print("=== BUSINESS INSIGHTS ===")
print(f"1. R&D Spend coefficient: {coefficients[2]:.2f}")
print("   - Every $1 increase in R&D spending increases profit by ${:.2f}".format(coefficients[2]))
print(f"2. Marketing Spend coefficient: {coefficients[4]:.2f}")
print("   - Every $1 increase in Marketing spending increases profit by ${:.2f}".format(coefficients[4]))
print(f"3. Administration coefficient: {coefficients[3]:.2f}")
print("   - Every $1 increase in Administration spending changes profit by ${:.2f}".format(coefficients[3]))

most_important_feature = feature_names[np.argmax(np.abs(coefficients))]
print(f"\n4. Most influential feature: {most_important_feature}")
print(f"   - Has the highest absolute coefficient value: {np.max(np.abs(coefficients)):.2f}")

In [ ]:
# Additional analysis: Prediction confidence intervals and summary
print("=== FINAL SUMMARY ===")
print(f"Model Performance: R² = {r2:.4f} ({r2*100:.1f}% variance explained)")
print(f"Average Prediction Error: ${mae:,.2f}")
print(f"Model Equation: Profit = {regressor.intercept_:.2f}", end="")
for i, coef in enumerate(regressor.coef_):
    print(f" + {coef:.2f}*X{i+1}", end="")
print()

# Show best and worst predictions
best_prediction = np.argmin(np.abs(y_test - y_pred))
worst_prediction = np.argmax(np.abs(y_test - y_pred))

print(f"\nBest Prediction:")
print(f"  Actual: ${y_test[best_prediction]:,.2f}, Predicted: ${y_pred[best_prediction]:,.2f}")
print(f"  Error: ${abs(y_test[best_prediction] - y_pred[best_prediction]):,.2f}")

print(f"\nWorst Prediction:")
print(f"  Actual: ${y_test[worst_prediction]:,.2f}, Predicted: ${y_pred[worst_prediction]:,.2f}")
print(f"  Error: ${abs(y_test[worst_prediction] - y_pred[worst_prediction]):,.2f}")

print(f"\nConclusion: This Multiple Linear Regression model explains {r2*100:.1f}% of profit variation")
print("and can be used for business decision-making with reasonable accuracy.")